In [1]:
import os
import sys
import numpy as np
import pandas as pd
from tqdm import tqdm

sys.path.append(os.path.abspath("../../")) ; from EPF import variables
sys.path.insert(0, os.path.abspath("../../../Generic-Parallel-Compute-Helper/")); from parallel_compute import *

import lightgbm as lgb
import json
import joblib

### High-accuracy NEM price forecaster.

#### Method

Spike-aware direct multi-horizon LightGBM ensemble with seven components
per forecast horizon:

  1. BASE L1 model   – regression_l1 on arcsinh(clip(y, p97)). Clipping the
                       target at the 97th percentile removes extreme-spike
                       noise from the base model's loss, giving it a clean
                       training signal for the majority of intervals.

  2. BASE L2 model   – regression (MSE) on arcsinh(y), uncapped target.
                       Blending L1+L2 (Lago et al. 2021) reduces both MAE
                       and RMSE simultaneously. Per-horizon blend weight α
                       tuned on the validation set.

  3. SPIKE classifier – binary LightGBM estimating P(price > SPIKE_THRESHOLD).
                        Trained with scale_pos_weight to handle class imbalance.

  4. SPIKE regressor  – regression_l1 on arcsinh(y) (full range, no clipping).
                        Spike rows receive 10x sample weight.

  5. SPIKE quantile   – Predicts the 90th percentile for conservative spike ceilings. 
                        P90 quantile regression on arcsinh(y) for a
                        conservative spike ceiling estimate.

  6. DIP classifier   – binary LightGBM estimating P(price < SPIKE_THRESHOLD).
                        Trained with scale_pos_weight to handle class imbalance.

  7. DIP regressor    – regression_l1 on arcsinh(y) (full range, no clipping).
                        DIP rows receive 7x sample weight.

  8. DIP quantile   –   Predicts the 10th percentile for conservative dip floors. 
                        P10 quantile regression on arcsinh(y) for a
                        conservative dip floor estimate.



Inputs

In [2]:
y_train_individual_clipped_transformed = pd.read_parquet("Data/1_derived_targets/y_train_individual_clipped_transformed.parquet")
y_validation_individual_clipped_transformed = pd.read_parquet("Data/1_derived_targets/y_validation_individual_clipped_transformed.parquet")
y_train_individual_unclipped_transformed = pd.read_parquet("Data/1_derived_targets/y_train_individual_unclipped_transformed.parquet")
y_validation_individual_unclipped_transformed = pd.read_parquet("Data/1_derived_targets/y_validation_individual_unclipped_transformed.parquet")
positive_spike_labels_train = pd.read_parquet("Data/1_derived_targets/positive_spike_labels_train.parquet")
positive_spike_labels_validate = pd.read_parquet("Data/1_derived_targets/positive_spike_labels_validate.parquet")
negative_spike_labels_train = pd.read_parquet("Data/1_derived_targets/negative_spike_labels_train.parquet")
negative_spike_labels_validate = pd.read_parquet("Data/1_derived_targets/negative_spike_labels_validate.parquet")
y_train_loss_weights = pd.read_parquet("Data/1_derived_targets/y_train_loss_weights.parquet")
y_train_full_range_loss_weights = pd.read_parquet("Data/1_derived_targets/y_train_full_range_loss_weights.parquet")
y_train_positive_spike_loss_weights = pd.read_parquet("Data/1_derived_targets/y_train_positive_spike_loss_weights.parquet")
y_train_negative_spike_loss_weights = pd.read_parquet("Data/1_derived_targets/y_train_negative_spike_loss_weights.parquet")

In [3]:
# Load hyperparameters
with open("Data/2_hyper_parameters/hyperparameters.json", "r", encoding="utf-8") as f:
    MODEL_HYPERPARAMETERS = json.load(f)

full_range_regressor_clipped_MAE_loss_params = MODEL_HYPERPARAMETERS["full_range_regressor_clipped_MAE_loss"]
full_range_regressor_unclipped_RMSE_loss_params = MODEL_HYPERPARAMETERS["full_range_regressor_unclipped_RMSE_loss"]
positive_spike_classifier_binary_loss_params = MODEL_HYPERPARAMETERS["positive_spike_classifier_binary_loss"]
positive_spike_regressor_unclipped_mae_loss_params = MODEL_HYPERPARAMETERS["positive_spike_regressor_unclipped_mae_loss"]
positive_spike_regressor_unclipped_quantile_loss_params = MODEL_HYPERPARAMETERS["positive_spike_regressor_unclipped_quantile_loss"]
negative_spike_classifer_unclipped_binary_loss_params = MODEL_HYPERPARAMETERS["negative_spike_classifer_unclipped_binary_loss"]
negative_spike_regressor_unclipped_mae_loss_params = MODEL_HYPERPARAMETERS["negative_spike_regressor_unclipped_mae_loss"]
negative_spike_regressor_unclipped_quantile_loss_params = MODEL_HYPERPARAMETERS["negative_spike_regressor_unclipped_quantile_loss"]

Core logic

In [4]:
# LGBM model setup
EARLY_STOPPING_ROUNDS = 75

def _train_single_model(X_train, y_train, X_validate, y_validate, loss_weights, hyperparameters):
    params = dict(hyperparameters)
    params["n_jobs"] = 1
    params["num_threads"] = 1
    model = lgb.LGBMRegressor(**params)

    model.fit(
        X_train,
        y_train,
        sample_weight=loss_weights,
        eval_set=[(X_validate, y_validate)],
        callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)],
    )
    return model

def _train_single_task(item: dict):
    horizon = item["horizon"]
    model_name = item["model_name"]
    hyperparameters = item["hyperparameters"]
    y_train_key = item["y_train_key"]
    y_validate_key = item["y_validate_key"]
    weight_key = item["weight_key"]

    h = f"h{horizon}"
    X_train_h = HORIZON_CACHE[h]["X_train"]
    X_validate_h = HORIZON_CACHE[h]["X_validate"]
    vectors = TARGET_CACHE[h]

    model = _train_single_model(
        X_train=X_train_h,
        y_train=vectors[y_train_key],
        X_validate=X_validate_h,
        y_validate=vectors[y_validate_key],
        loss_weights=vectors[weight_key],
        hyperparameters=hyperparameters,
    )

    filename = f"h{horizon:02d}_{model_name}.joblib"
    model_path = os.path.join(variables.TRAINED_MODELS_PATH, filename)
    joblib.dump(model, model_path)

    return {
        "horizon": horizon,
        "model_name": model_name,
        "model_path": model_path,
    }

In [5]:
model_specs = [
    ("full_range_regressor_clipped_MAE_loss", full_range_regressor_clipped_MAE_loss_params, "y_train_clipped", "y_validate_clipped", "w_full_range"),
    ("full_range_regressor_unclipped_RMSE_loss", full_range_regressor_unclipped_RMSE_loss_params, "y_train_unclipped", "y_validate_unclipped", "w_full_range"),
    ("positive_spike_classifier_binary_loss", positive_spike_classifier_binary_loss_params, "y_train_positive_label", "y_validate_positive_label", "w_default"),
    ("positive_spike_regressor_unclipped_mae_loss", positive_spike_regressor_unclipped_mae_loss_params, "y_train_unclipped", "y_validate_unclipped", "w_full_range"),
    ("positive_spike_regressor_unclipped_quantile_loss", positive_spike_regressor_unclipped_quantile_loss_params, "y_train_unclipped", "y_validate_unclipped", "w_positive"),
    ("negative_spike_classifer_unclipped_binary_loss", negative_spike_classifer_unclipped_binary_loss_params, "y_train_negative_label", "y_validate_negative_label", "w_default"),
    ("negative_spike_regressor_unclipped_mae_loss", negative_spike_regressor_unclipped_mae_loss_params, "y_train_unclipped", "y_validate_unclipped", "w_negative"),
    ("negative_spike_regressor_unclipped_quantile_loss", negative_spike_regressor_unclipped_quantile_loss_params, "y_train_unclipped", "y_validate_unclipped", "w_negative"),
]

In [6]:
feature_data = pd.read_parquet(variables.FEATURES_DATASET_PATH)

In [7]:
feature_data[:-5]

,nsw_price_asinh_lag_1,nsw_price_asinh_lag_2,nsw_price_asinh_lag_4,nsw_price_asinh_lag_12,nsw_price_asinh_lag_48,nsw_price_asinh_lag_96,nsw_price_asinh_lag_336,nsw_price_asinh_lag_335,nsw_price_asinh_lag_337,nsw_price_asinh_rmean_48,...,vic_price_vs_nsw_price_spread_lag1,vic_price_vs_qld_price_spread,vic_price_vs_qld_price_spread_lag1,vic_price_vs_sa_price_spread,vic_price_vs_sa_price_spread_lag1,multi_region_spike,region_spike_count,doy_sin,doy_cos,sa_spread_live
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:05:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,10.339350,NaN,-12.968930,NaN,0.0,0.0,0.017202,0.999852,0.133291
2018-01-01 00:10:00,0.810427,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.399640,11.636780,10.339350,-15.080300,-12.968930,0.0,0.0,0.017202,0.999852,0.162055
2018-01-01 00:15:00,0.849487,0.810427,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.196190,12.244390,11.636780,-16.358509,-15.080300,0.0,0.0,0.017202,0.999852,0.173223
2018-01-01 00:20:00,0.854923,0.849487,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.050520,11.096110,12.244390,-13.927330,-16.358509,0.0,0.0,0.017202,0.999852,0.142040
2018-01-01 00:25:00,0.827334,0.854923,0.810427,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.324450,11.124200,11.096110,-15.698750,-13.927330,0.0,0.0,0.017202,0.999852,0.159641
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-31 23:15:00,0.663137,0.545535,0.543191,0.645833,0.640081,0.541019,0.726974,0.773916,0.782782,0.619180,...,-70.201897,-62.450001,-77.750160,0.180610,0.150640,0.0,0.0,-0.004300,0.999991,-0.535974
2025-12-31 23:20:00,0.545535,0.663137,0.545535,0.655941,0.718820,0.605098,0.773916,0.805145,0.726974,0.615569,...,-56.020000,-60.809448,-62.450001,0.347180,0.180610,0.0,0.0,-0.004300,0.999991,-0.527120
2025-12-31 23:25:00,0.545535,0.545535,0.545535,0.559808,0.688672,0.510393,0.805145,0.786218,0.773916,0.612587,...,-54.840000,-49.551670,-60.809448,1.758460,0.347180,0.0,0.0,-0.004300,0.999991,-0.446500


In [8]:
X_train = feature_data[(feature_data.index >= variables.TRAIN_START) & (feature_data.index <= variables.VALID_START)]
X_validate = feature_data[(feature_data.index > variables.VALID_START) & (feature_data.index <= variables.TEST_START)]
X_test = feature_data[feature_data.index > variables.TEST_START]

features_optimal_amount = pd.read_parquet(variables.FEATURES_OPTIMAL_AMOUNT_PATH)

In [9]:
display(feature_data.shape)
display(X_train.shape)
display(X_validate.shape)
display(X_test.shape)

(841536, 637)

(578305, 637)

(52992, 637)

(105120, 637)

In [10]:
features_optimal_amount[:10]

,feature,h1,h2,h3,h4,h5,h6,h7,h8,h9,...,h87,h88,h89,h90,h91,h92,h93,h94,h95,h96
0,nsw_price_asinh_lag_1,False,False,False,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
1,nsw_price_asinh_lag_2,False,False,False,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
2,nsw_price_asinh_lag_4,False,False,False,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
3,nsw_price_asinh_lag_12,False,False,False,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
4,nsw_price_asinh_lag_48,False,False,False,False,False,False,False,True,True,...,False,False,False,False,False,False,False,False,False,False
5,nsw_price_asinh_lag_96,False,False,False,False,False,False,False,True,True,...,False,False,False,False,False,False,False,False,False,False
6,nsw_price_asinh_lag_336,False,False,False,False,False,False,False,True,True,...,False,False,False,False,False,False,False,False,False,False
7,nsw_price_asinh_lag_335,False,False,False,False,False,False,False,True,True,...,False,False,False,False,False,False,False,False,False,False
8,nsw_price_asinh_lag_337,False,False,False,False,False,False,False,True,True,...,False,False,False,False,False,False,False,False,False,False
9,nsw_price_asinh_rmean_48,False,False,False,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False


In [11]:
horizon_list = list(range(1, variables.HORIZON_COUNT + 1))
HORIZON_CACHE = {}
TARGET_CACHE = {}
TRAIN_TASKS = []

for horizon in horizon_list:
    h = f"h{horizon}"
    features_for_this_horizon = features_optimal_amount.loc[
        features_optimal_amount[h] == True, "feature"
    ].tolist()

    HORIZON_CACHE[h] = {
        "X_train": np.ascontiguousarray(X_train[features_for_this_horizon].to_numpy(dtype=np.float32)),
        "X_validate": np.ascontiguousarray(X_validate[features_for_this_horizon].to_numpy(dtype=np.float32)),
    }

    TARGET_CACHE[h] = {
        "y_train_clipped": np.ascontiguousarray(y_train_individual_clipped_transformed[h].to_numpy(dtype=np.float32)),
        "y_validate_clipped": np.ascontiguousarray(y_validation_individual_clipped_transformed[h].to_numpy(dtype=np.float32)),
        "y_train_unclipped": np.ascontiguousarray(y_train_individual_unclipped_transformed[h].to_numpy(dtype=np.float32)),
        "y_validate_unclipped": np.ascontiguousarray(y_validation_individual_unclipped_transformed[h].to_numpy(dtype=np.float32)),
        "y_train_positive_label": np.ascontiguousarray(positive_spike_labels_train[h].to_numpy(dtype=np.float32)),
        "y_validate_positive_label": np.ascontiguousarray(positive_spike_labels_validate[h].to_numpy(dtype=np.float32)),
        "y_train_negative_label": np.ascontiguousarray(negative_spike_labels_train[h].to_numpy(dtype=np.float32)),
        "y_validate_negative_label": np.ascontiguousarray(negative_spike_labels_validate[h].to_numpy(dtype=np.float32)),
        "w_default": np.ascontiguousarray(y_train_loss_weights[h].to_numpy(dtype=np.float32)),
        "w_full_range": np.ascontiguousarray(y_train_full_range_loss_weights[h].to_numpy(dtype=np.float32)),
        "w_positive": np.ascontiguousarray(y_train_positive_spike_loss_weights[h].to_numpy(dtype=np.float32)),
        "w_negative": np.ascontiguousarray(y_train_negative_spike_loss_weights[h].to_numpy(dtype=np.float32)),
    }

    for model_name, hyperparameters, y_train_key, y_validate_key, weight_key in model_specs:
        TRAIN_TASKS.append({
            "horizon": horizon,
            "model_name": model_name,
            "hyperparameters": hyperparameters,
            "y_train_key": y_train_key,
            "y_validate_key": y_validate_key,
            "weight_key": weight_key,
        })

In [12]:
print("model count:", len(TRAIN_TASKS))

model count: 768


95% 1 min


In [13]:
import gc

os.makedirs(variables.TRAINED_MODELS_PATH, exist_ok=True)

# Use helper parallelism with one model per task for smoother progress updates.
trained_models = run_parallel(function=_train_single_task, data=TRAIN_TASKS)

gc.collect()
print(f"Finished training {len(trained_models)} models.")

os=linux cpu_count=48 max_assigned_workers=45


Processing..: 100%|██████████| 768/768 [2:28:33<00:00, 11.61s/item]  


Finished training 768 models.


Export

In [14]:
for m in trained_models:
    # Skip models already saved during training.
    if "model" not in m:
        continue

    filename = f"h{m['horizon']:02d}_{m['model_name']}.joblib"
    joblib.dump(m["model"], f"{variables.TRAINED_MODELS_PATH}/{filename}")